# Examen 2
> Moctezuma Ramirez Diego Rafael

## Carga de modulos

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
import pandas as pd
import numpy as np

import plotly.express as px

# Data preprocessing
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling
from sklearn.model_selection import cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, Lars, Lasso, BayesianRidge, SGDRegressor

#pd.set_option('display.float_format', lambda x: '%.6f' % x)

## Funciones auxiliares

In [3]:
def frecuencias(df, caracteristicas):
    for caracteristica in caracteristicas:
        print(f"Caracteristica: {caracteristica}")
        abs_ = df[caracteristica].value_counts(dropna=False).to_frame().rename(columns={"count": "Frecuencia absoluta"})
        rel_ = df[caracteristica].value_counts(dropna=False, normalize= True).to_frame().rename(columns={"proportion": "Frecuencia relativa"})
        freq = abs_.join(rel_)
        freq["Frecuencia Acumulada"] = freq["Frecuencia absoluta"].cumsum()
        freq["Acumulada %"] = freq["Frecuencia relativa"].cumsum()
        freq["Frecuencia absoluta"] = freq["Frecuencia absoluta"].map(lambda x: f"{x:,.0f}")
        freq["Frecuencia relativa"] = freq["Frecuencia relativa"].map(lambda x: f"{x:,.2%}")
        freq["Frecuencia Acumulada"] = freq["Frecuencia Acumulada"].map(lambda x: f"{x:,.0f}")
        freq["Acumulada %"] = freq["Acumulada %"].map(lambda x: f"{x:,.2%}")
        display(freq)

## Carga de datos con pandas

In [4]:
df = pd.read_csv("DataRegresion/train_PAY_AMT3.csv",sep='|')

In [5]:
df.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_4,PAY_5,PAY_6,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
41,9882,60000.0,1,2,2,25,0,0,0,58583.0,28301.0,29301.0,14400.0,600.0,1000.0,300.0,28154.0
3964,5976,20000.0,2,2,2,30,0,0,0,18979.0,7886.0,19786.0,20025.0,1000.0,19000.0,6000.0,1000.0
2803,16947,30000.0,1,2,2,34,4,3,2,24633.0,23961.0,23289.0,22779.0,0.0,0.0,0.0,3400.0
19,14956,390000.0,2,1,2,32,0,0,0,15204.0,13529.0,14139.0,14496.0,1500.0,1500.0,519.0,7000.0
5116,17432,50000.0,2,2,1,28,2,0,0,15377.0,12590.0,11209.0,7448.0,0.0,3023.0,5000.0,4000.0


In [6]:
df.dtypes

CUSTOMER_ID      int64
LIMIT_BAL      float64
SEX              int64
EDUCATION        int64
MARRIAGE         int64
AGE              int64
PAY_4            int64
PAY_5            int64
PAY_6            int64
BILL_AMT3      float64
BILL_AMT4      float64
BILL_AMT5      float64
BILL_AMT6      float64
PAY_AMT3       float64
PAY_AMT4       float64
PAY_AMT5       float64
PAY_AMT6       float64
dtype: object

In [7]:
df.isna().sum()

CUSTOMER_ID    0
LIMIT_BAL      0
SEX            0
EDUCATION      0
MARRIAGE       0
AGE            0
PAY_4          0
PAY_5          0
PAY_6          0
BILL_AMT3      0
BILL_AMT4      0
BILL_AMT5      0
BILL_AMT6      0
PAY_AMT3       0
PAY_AMT4       0
PAY_AMT5       0
PAY_AMT6       0
dtype: int64

## EDA

### Filtrado de variables

In [8]:
ls_discretas = ["SEX","EDUCATION","MARRIAGE","PAY_4","PAY_5","PAY_6"]
ls_continuas = ["BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6","PAY_AMT4","PAY_AMT5","PAY_AMT6","LIMIT_BAL","AGE"]
ls_drop = ["CUSTOMER_ID"]

target = "PAY_AMT3"

In [9]:
df_customer_ids = df["CUSTOMER_ID"]
df.drop(columns=ls_drop, inplace=True)

### EDA Discreto

In [10]:
frecuencias(df, ls_discretas)

Caracteristica: SEX


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
SEX,,,,
2,"3,411",60.64%,"3,411",60.64%
1,"2,214",39.36%,"5,625",100.00%


Caracteristica: EDUCATION


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
EDUCATION,,,,
2,"2,667",47.41%,"2,667",47.41%
1,"2,011",35.75%,"4,678",83.16%
3,867,15.41%,"5,545",98.58%
5,52,0.92%,"5,597",99.50%
4,17,0.30%,"5,614",99.80%
6,9,0.16%,"5,623",99.96%
0,2,0.04%,"5,625",100.00%


Caracteristica: MARRIAGE


...

,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_4,,,,
0,"3,100",55.11%,"3,100",55.11%
-1,"1,042",18.52%,"4,142",73.64%
-2,810,14.40%,"4,952",88.04%
2,604,10.74%,"5,556",98.77%
3,40,0.71%,"5,596",99.48%
4,17,0.30%,"5,613",99.79%
7,7,0.12%,"5,620",99.91%
5,5,0.09%,"5,625",100.00%


Caracteristica: PAY_5


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_5,,,,
0,"3,196",56.82%,"3,196",56.82%
-1,"1,024",18.20%,"4,220",75.02%
-2,846,15.04%,"5,066",90.06%
2,500,8.89%,"5,566",98.95%
3,32,0.57%,"5,598",99.52%
4,20,0.36%,"5,618",99.88%
7,6,0.11%,"5,624",99.98%
6,1,0.02%,"5,625",100.00%


Caracteristica: PAY_6


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_6,,,,
0,"3,052",54.26%,"3,052",54.26%
-1,"1,068",18.99%,"4,120",73.24%
-2,902,16.04%,"5,022",89.28%
2,553,9.83%,"5,575",99.11%
3,32,0.57%,"5,607",99.68%
4,8,0.14%,"5,615",99.82%
7,6,0.11%,"5,621",99.93%
6,2,0.04%,"5,623",99.96%
5,2,0.04%,"5,625",100.00%


### EDA Continuas

In [11]:
for caracteristica in ls_continuas:
    fig = px.histogram(df, x=caracteristica, nbins=50, title=f"Histograma de {caracteristica}",template = "plotly_dark")
    fig.show()

### Feature Engineering

In [12]:
df['TOTAL_DEBT'] = df[['BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['USAGE_RATIO'] = df['TOTAL_DEBT'] / df['LIMIT_BAL']
df['TOTAL_PAYMENTS'] = df[['PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['PAY_AMT_MEAN'] = df[['PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].mean(axis=1)
df['PAY_STA_MEAN'] = df[['PAY_4', 'PAY_5', 'PAY_6']].mean(axis=1).round(2)

#### Normalización

> EDUCATION

In [13]:
redundantes_e = [0,5,6]
df['EDUCATION'] = df['EDUCATION'].replace(redundantes_e, 4)

In [14]:
df["EDUCATION"].value_counts(True)

EDUCATION
2    0.474133
1    0.357511
3    0.154133
4    0.014222
Name: proportion, dtype: float64

> MARRIAGE

In [15]:
redundantes_m = [0]
df['MARRIAGE'] = df['MARRIAGE'].replace(redundantes_m, 3)

In [16]:
df["MARRIAGE"].value_counts(True)

MARRIAGE
2    0.540978
1    0.443200
3    0.015822
Name: proportion, dtype: float64

In [17]:
df[ls_discretas].sample(5)

,SEX,EDUCATION,MARRIAGE,PAY_4,PAY_5,PAY_6
2379,2,1,1,-2,-1,-1
3073,2,1,2,0,0,0
2964,1,2,1,2,2,2
4189,2,1,2,2,2,2
2406,1,2,1,0,0,0


### Data Cleaning

In [18]:
shape_out = df.shape

In [19]:
df[ls_continuas].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT4,PAY_AMT5,PAY_AMT6,LIMIT_BAL,AGE
count,5.625000e+03,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000
mean,4.818763e+04,44132.105067,41366.440533,40102.049067,4985.842844,4823.514133,5183.265778,167488.000000,35.248711
std,7.333285e+04,65962.517110,62289.294777,61087.413931,14925.386577,15507.689797,16741.595055,129776.181548,9.071012
min,-1.095100e+04,-3849.000000,-19205.000000,-57060.000000,0.000000,0.000000,0.000000,10000.000000,21.000000
1%,-1.154400e+02,-123.800000,-283.040000,-371.760000,0.000000,0.000000,0.000000,10000.000000,22.000000
5%,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20000.000000,23.000000
50%,2.066200e+04,19298.000000,18826.000000,18175.000000,1650.000000,1576.000000,1572.000000,140000.000000,34.000000
95%,1.917342e+05,175085.400000,165629.200000,163166.800000,18000.400000,15000.000000,18243.800000,420000.000000,53.000000
99%,3.353153e+05,323586.000000,287650.840000,285481.400000,66643.280000,64505.280000,76989.880000,500000.000000,59.000000
max,1.664089e+06,891586.000000,927171.000000,961664.000000,313094.000000,388071.000000,377000.000000,1000000.000000,72.000000


In [20]:
dic_outliers = {}
for caracteristica in ls_continuas:
    aux = df[caracteristica].quantile(0.99) + df[caracteristica].std() * 4
    max_val = df[caracteristica].max()
    if max_val > aux:
        dic_outliers[caracteristica] = df[caracteristica].quantile(0.99)

In [21]:
dic_outliers

{'BILL_AMT3': np.float64(335315.3200000003),
 'BILL_AMT4': np.float64(323586.0000000009),
 'BILL_AMT5': np.float64(287650.84000000014),
 'BILL_AMT6': np.float64(285481.4000000002),
 'PAY_AMT4': np.float64(66643.28000000038),
 'PAY_AMT5': np.float64(64505.28000000001),
 'PAY_AMT6': np.float64(76989.88000000022)}

In [22]:
for caracteristica, perc in dic_outliers.items():
    df = df[(df[caracteristica] <= perc) | (df[caracteristica].isna())]

In [23]:
df.shape[0] / shape_out[0] 

0.9585777777777778

#### Variables altamente vacías

In [24]:
df.isna().mean().sort_values()

LIMIT_BAL         0.0
TOTAL_PAYMENTS    0.0
USAGE_RATIO       0.0
TOTAL_DEBT        0.0
PAY_AMT6          0.0
PAY_AMT5          0.0
PAY_AMT4          0.0
PAY_AMT3          0.0
BILL_AMT6         0.0
PAY_AMT_MEAN      0.0
BILL_AMT5         0.0
BILL_AMT3         0.0
PAY_6             0.0
PAY_5             0.0
PAY_4             0.0
AGE               0.0
MARRIAGE          0.0
EDUCATION         0.0
SEX               0.0
BILL_AMT4         0.0
PAY_STA_MEAN      0.0
dtype: float64

#### Variables unarias

In [25]:
df.nunique()

LIMIT_BAL           67
SEX                  2
EDUCATION            4
MARRIAGE             3
AGE                 49
PAY_4                8
PAY_5                8
PAY_6                9
BILL_AMT3         4534
BILL_AMT4         4442
BILL_AMT5         4387
BILL_AMT6         4304
PAY_AMT3          2147
PAY_AMT4          2017
PAY_AMT5          1955
PAY_AMT6          1955
TOTAL_DEBT        4812
USAGE_RATIO       4931
TOTAL_PAYMENTS    3259
PAY_AMT_MEAN      3259
PAY_STA_MEAN        21
dtype: int64

In [26]:
df.std()

LIMIT_BAL         123855.953156
SEX                    0.487884
EDUCATION              0.732206
MARRIAGE               0.526787
AGE                    9.076203
PAY_4                  1.167796
PAY_5                  1.123884
PAY_6                  1.149946
BILL_AMT3          56862.346286
BILL_AMT4          52925.700354
BILL_AMT5          49237.565735
BILL_AMT6          48281.343176
PAY_AMT3           12845.276363
PAY_AMT4            6616.185159
PAY_AMT5            6130.456294
PAY_AMT6            7296.287721
TOTAL_DEBT        148154.670878
USAGE_RATIO            1.021040
TOTAL_PAYMENTS     14517.911704
PAY_AMT_MEAN        4839.303901
PAY_STA_MEAN           1.061114
dtype: float64

### Selección mejores variables

In [27]:
kb = SelectKBest(k="all", score_func=f_regression)

In [28]:
columnas = df.columns.to_list()
columnas.remove(target)
X = df[columnas]
y = df[target]

In [29]:
kb.fit(X, y)

,score_func,<function f_r...t 0x143a40160>
,k,'all'


In [30]:
df_scores = pd.DataFrame(data=zip(X.columns, kb.scores_), columns=["feature", "score"]).set_index("feature").sort_values(by="score", ascending=False)

In [31]:
df_scores.head(20)

,score
feature,
BILL_AMT4,499.287869
PAY_AMT_MEAN,417.047288
TOTAL_PAYMENTS,417.047288
TOTAL_DEBT,404.666404
BILL_AMT5,361.186747
BILL_AMT6,318.370250
PAY_AMT4,245.378756
PAY_AMT5,198.837717
PAY_AMT6,192.152169


In [32]:
fig = px.bar(df_scores, template = "plotly_dark")
fig.show()

## Modelado

### Entrenamiento del modelo

#### Standard Escaler

In [33]:
ss = StandardScaler()

#### Regresión Lineal

In [34]:
lg = LinearRegression()

#### LARS

In [35]:
lrs = Lars()

#### LASSO

In [36]:
lasso = Lasso()

#### Ridge

In [37]:
ridge = Ridge()

#### Elastic Net

In [38]:
elan = ElasticNet()

#### Red Bayesiana

In [39]:
redbay = BayesianRidge()

#### Gradiente Estocástico Descendiente

In [40]:
ged = SGDRegressor()

### Comparación de modelos

#### Usando selectKBest

In [41]:
modelos = {
    "Regresion Lineal": lg,
    "Lars": lrs,
    "Lasso": lasso,
    "Ridge": ridge,
    "Elastic Net": elan,
    "Red Bayesiana": redbay,
    "Gradiente Estocástico": ged
}

In [42]:
df_kbest_model = pd.DataFrame(columns=["Regresion Lineal", "Lars", "Lasso", "Ridge", "Elastic Net", "Red Bayesiana", "Gradiente Estocástico"])
X_train, X_test, y_train, y_test = train_test_split(X, y)

y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [43]:
transformer = ColumnTransformer([
    ("scaler", ss, ls_continuas),
    ("dummies", OneHotEncoder(handle_unknown="ignore"), ls_discretas)
])

In [44]:
for k in range(5, 41, 5):
    resultados_k = {}   

    for nombre, modelo in modelos.items():
        pipe = Pipeline([
            ("prep", transformer),
            ("kbest", SelectKBest(f_regression, k=k)),
            ("modelo", modelo)
        ])
        pipe.fit(X_train, y_train)
        score = pipe.score(X_test, y_test)
        resultados_k[nombre] = score

    df_kbest_model.loc[k] = resultados_k

In [45]:
df_kbest_model

,Regresion Lineal,Lars,Lasso,Ridge,Elastic Net,Red Bayesiana,Gradiente Estocástico
5,0.190759,1.907592e-01,0.190751,0.190791,0.127950,0.190848,0.186559
10,0.451181,4.511811e-01,0.451125,0.450694,0.179105,0.450457,0.447688
15,0.452719,4.527192e-01,0.452673,0.452305,0.181832,0.451853,0.440804
20,0.447926,4.479261e-01,0.448087,0.447639,0.183363,0.447176,0.445538
25,0.450221,4.502215e-01,0.450460,0.450063,0.183683,0.449657,0.443772
30,0.451366,4.513661e-01,0.451588,0.451115,0.183682,0.450546,0.449288
35,0.450516,-1.667063e+03,0.450743,0.450374,0.183827,0.449653,0.444310
40,0.450438,-4.200499e+09,0.450578,0.450237,0.184043,0.449281,0.443321


In [46]:
max_lr = [df_kbest_model["Regresion Lineal"].idxmax(),df_kbest_model["Regresion Lineal"].max()]
max_lars = [df_kbest_model["Lars"].idxmax(),df_kbest_model["Lars"].max()]
max_lasso = [df_kbest_model["Lasso"].idxmax(),df_kbest_model["Lasso"].max()]
max_ridge = [df_kbest_model["Ridge"].idxmax(),df_kbest_model["Ridge"].max()]
max_elnet = [df_kbest_model["Elastic Net"].idxmax(),df_kbest_model["Elastic Net"].max()]
max_redbay = [df_kbest_model["Red Bayesiana"].idxmax(),df_kbest_model["Red Bayesiana"].max()]
max_ged = [df_kbest_model["Gradiente Estocástico"].idxmax(),df_kbest_model["Gradiente Estocástico"].max()]

In [47]:
df_modelos = pd.DataFrame(columns=["modelo","mejor_k","mejor_score"])

df_modelos.loc[0] = ["Linear Regression", max_lr[0], max_lr[1]]
df_modelos.loc[1] = ["LARS", max_lars[0], max_lars[1]]
df_modelos.loc[2] = ["Lasso", max_lasso[0], max_lasso[1]]
df_modelos.loc[3] = ["Ridge", max_ridge[0], max_ridge[1]]
df_modelos.loc[4] = ["Elastic Net", max_elnet[0], max_elnet[1]]
df_modelos.loc[5] = ["Red Bayesiana", max_redbay[0], max_redbay[1]]
df_modelos.loc[6] = ["Gradiente Estocástico", max_ged[0], max_ged[1]]

In [48]:
df_modelos.sort_values(by="mejor_score", ascending=False)

,modelo,mejor_k,mejor_score
1,LARS,15,0.452719
0,Linear Regression,15,0.452719
2,Lasso,15,0.452673
3,Ridge,15,0.452305
5,Red Bayesiana,15,0.451853
6,Gradiente Estocástico,30,0.449288
4,Elastic Net,40,0.184043


##### Cross validation 

In [49]:
df_crossv = pd.DataFrame(columns = ["modelo","f1","f2","f3","f4","f5","mean","std"])

In [50]:
dic_cross = {
    "Regresion lineal": [lg, max_lr[0]],
    "LARS": [lrs, max_lars[0]],
    "Lasso": [lasso, max_lasso[0]],
    "Ridge": [ridge, max_ridge[0]],
    "Elastic Net": [elan, max_elnet[0]],
    "Red Bayesiana": [redbay, max_redbay[0]],
    "Gradiente Estocástico": [ged, max_ged[0]]
}

In [51]:
for nombre_modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(score_func=f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    cross = cross_val_score(estimator=pipe,X=X,y=y.values.ravel(),cv=5)

    df_crossv.loc[len(df_crossv)] = [nombre_modelo,cross[0],cross[1],cross[2],cross[3],cross[4],cross.mean(),cross.std()    ]

In [52]:
df_crossv

,modelo,f1,f2,f3,f4,f5,mean,std
0,Regresion lineal,0.326121,0.387597,0.189622,0.446397,0.283503,0.326648,0.087969
1,LARS,0.326121,0.387597,0.189622,0.446441,0.283503,0.326657,0.087981
2,Lasso,0.326246,0.387453,0.189914,0.446257,0.283729,0.326720,0.087798
3,Ridge,0.326936,0.386973,0.191883,0.445625,0.284700,0.327223,0.086849
4,Elastic Net,0.192605,0.135390,0.220932,0.158077,0.168628,0.175126,0.029378
5,Red Bayesiana,0.327459,0.386400,0.193167,0.444714,0.285612,0.327471,0.086033
6,Gradiente Estocástico,0.343306,0.390363,0.203884,0.440279,0.308470,0.337260,0.080109


##### Comparación de cross validations

In [53]:
df_crossv[["modelo","mean"]].sort_values(by="mean", ascending=False)

,modelo,mean
6,Gradiente Estocástico,0.337260
5,Red Bayesiana,0.327471
3,Ridge,0.327223
2,Lasso,0.326720
1,LARS,0.326657
0,Regresion lineal,0.326648
4,Elastic Net,0.175126


## Hiperparametrización

In [54]:
param_grids = {
    "Regresion lineal": {
        "modelo__fit_intercept": [True, False],
        "modelo__positive": [False, True]
    },

    "LARS": {
        "modelo__fit_intercept": [True, False],
        "modelo__n_nonzero_coefs": [5, 10, 20, 50, 100, np.inf]
    },

    "Lasso": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Ridge": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__solver": ["auto", "svd", "cholesky", "lsqr"]
    },

    "Elastic Net": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__l1_ratio": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Red Bayesiana": {
        "modelo__alpha_1": [1e-6, 1e-4, 1e-2],
        "modelo__alpha_2": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_1": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_2": [1e-6, 1e-4, 1e-2]
    },

    "Gradiente Estocástico": {
        "modelo__loss": ["squared_error", "huber"],
        "modelo__penalty": ["l2", "l1", "elasticnet"],
        "modelo__alpha": [0.0001, 0.001, 0.01],
        "modelo__learning_rate": ["constant", "optimal", "invscaling", "adaptive"],
        "modelo__eta0": [0.001, 0.01, 0.1],
        "modelo__max_iter": [1000, 5000, 10000]
    }
}

In [55]:
df_hyper = pd.DataFrame(columns=["Modelo", "Mejor score", "Std", "Parametros buenos"])

for modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    grid = param_grids[modelo]
    search = RandomizedSearchCV(estimator=pipe, param_distributions=grid, scoring="r2", n_iter=20, cv=5, n_jobs=-1, random_state=777)

    search.fit(X_train, y_train)
    df_hyper.loc[len(df_hyper)] = [modelo,search.best_score_,search.cv_results_['std_test_score'][search.best_index_],search.best_params_]

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.



/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning:


10 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/CD/lib/python3.10/site-package

In [56]:
df_hyper.sort_values(by="Mejor score", ascending=False)

,Modelo,Mejor score,Std,Parametros buenos
4,Elastic Net,0.325865,0.121499,"{'modelo__max_iter': 1000, 'modelo__l1_ratio':..."
6,Gradiente Estocástico,0.321926,0.138128,"{'modelo__penalty': 'l2', 'modelo__max_iter': ..."
3,Ridge,0.320897,0.126904,"{'modelo__solver': 'lsqr', 'modelo__alpha': 10.0}"
2,Lasso,0.319050,0.136263,"{'modelo__max_iter': 1000, 'modelo__alpha': 10.0}"
0,Regresion lineal,0.318764,0.138667,"{'modelo__positive': False, 'modelo__fit_inter..."
1,LARS,0.318764,0.138667,"{'modelo__n_nonzero_coefs': 20, 'modelo__fit_i..."
5,Red Bayesiana,0.318374,0.136697,"{'modelo__lambda_2': 0.01, 'modelo__lambda_1':..."


## Predicción

### Mejor modelo

#### Standard - Select K Best

In [57]:
nombre_modelo = df_crossv.loc[df_crossv["mean"].idxmax(), "modelo"]
info_modelo = dic_cross[nombre_modelo]
nombre_modelo

'Gradiente Estocástico'

In [58]:
modelo_ss = info_modelo[0]
mejor_k = info_modelo[1]

In [59]:
pipe_ss = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_ss)
])

In [60]:
pipe_ss.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


#### Hiperparametrización

In [61]:
df_hyper["Parametros buenos"][df_hyper["Modelo"] == "Elastic Net"].values[0]

{'modelo__max_iter': 1000, 'modelo__l1_ratio': 0.4, 'modelo__alpha': 0.01}

In [62]:
modelo_h = ElasticNet(max_iter=1000,l1_ratio=0.4, alpha=0.01)
mejor_k = max_elnet[0]

In [63]:
pipe_h = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_h)
])

In [64]:
pipe_h.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Carga de datos

In [65]:
df_predic = pd.read_csv("DataRegresion/val_PAY_AMT3.csv",sep='|')

In [66]:
df_predic

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_4,PAY_5,PAY_6,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT4,PAY_AMT5,PAY_AMT6,BILL_AMT3
0,14941,260000.0,1,1,2,36,-2,-2,-2,-73.0,-73.0,-73.0,0.0,0.0,0.0,427.0
1,14778,90000.0,2,2,1,49,0,0,0,46331.0,47494.0,48433.0,2000.0,1746.0,1941.0,45385.0
2,1377,220000.0,1,2,1,39,0,0,0,7412.0,7886.0,8358.0,600.0,600.0,600.0,6478.0
3,14681,360000.0,2,1,2,27,0,0,0,289355.0,274484.0,247998.0,10000.0,10000.0,10000.0,314534.0
4,17780,50000.0,1,2,1,44,2,0,0,12790.0,12743.0,17305.0,2000.0,5000.0,3000.0,50175.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870,28245,300000.0,1,3,1,56,-2,-2,-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1871,1419,230000.0,2,2,1,30,-1,0,0,25916.0,4219.0,7903.0,2219.0,4000.0,6000.0,2923.0
1872,935,60000.0,2,2,2,31,0,0,0,28187.0,28978.0,27119.0,1200.0,1500.0,1500.0,55775.0
1873,21360,160000.0,1,2,2,25,0,0,0,82926.0,84278.0,93641.0,2668.0,11259.0,2043.0,71405.0


### Normalización

> EDUCATION

In [67]:
redundantes_e = [0,5,6]
df_predic['EDUCATION'] = df_predic['EDUCATION'].replace(redundantes_e, 4)

> MARRIAGE

In [68]:
redundantes_m = [0]
df_predic['MARRIAGE'] = df_predic['MARRIAGE'].replace(redundantes_m, 3)

### Limpieza

In [69]:
shape_viejo = df_predic.shape

In [70]:
dic_outliers

{'BILL_AMT3': np.float64(335315.3200000003),
 'BILL_AMT4': np.float64(323586.0000000009),
 'BILL_AMT5': np.float64(287650.84000000014),
 'BILL_AMT6': np.float64(285481.4000000002),
 'PAY_AMT4': np.float64(66643.28000000038),
 'PAY_AMT5': np.float64(64505.28000000001),
 'PAY_AMT6': np.float64(76989.88000000022)}

In [71]:
for caracteristica, perc in dic_outliers.items():
    df_predic = df_predic[(df_predic[caracteristica] <= perc) | (df_predic[caracteristica].isna())]

In [72]:
df_predic.shape[0] / shape_viejo[0]

0.9605333333333334

### Predicción

In [73]:
ids = df_predic["CUSTOMER_ID"]
df_predic.drop(columns=ls_drop, inplace=True)

In [74]:
pred1 = pipe_ss.predict(df_predic)

In [75]:
pred2 = pipe_h.predict(df_predic)

In [76]:
df_resultado_cv = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred1
})

In [77]:
df_resultado_hp = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred2
})

In [82]:
df_resultado_cv.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT3.csv", index=False)

In [79]:
df_resultado_cv

,CARDHOLDER_ID,y_hat
0,14941,948.369167
1,14778,3761.509744
2,1377,1973.940906
3,14681,19302.359878
4,17780,-9052.698458
...,...,...
1870,28245,1804.129278
1871,1419,21261.839251
1872,935,-4833.748582
1873,21360,10246.302144


In [83]:
df_resultado_hp.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT3.csv", index=False)

In [81]:
df_resultado_hp

,CARDHOLDER_ID,y_hat
0,14941,1318.727341
1,14778,3353.172797
2,1377,1949.903368
3,14681,18254.246987
4,17780,-7961.710241
...,...,...
1870,28245,1645.040708
1871,1419,20151.075316
1872,935,-3981.869329
1873,21360,10546.385412
